# PhysIQ – Task 4: Tool Use Planning (Multi-Turn)

Given a 2D physics environment and a goal, interact with the simulation across up to 10 turns to achieve the goal by placing, pushing, or removing objects.

**Actions:**
```
ACTION: PLACE <object_type> AT <x> <y> ANGLE <degrees>
ACTION: PUSH <object_id> FORCE <newtons> DIRECTION <degrees>
ACTION: REMOVE <object_id>
```
**Scoring:** Goal achieved (0.6) + efficiency (0.2) + reasoning quality (0.2)

In [ ]:
# ── Cell 1: Setup & imports ──────────────────────────────────────────────────
import sys, os, json, re
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

In [ ]:
# ── Cell 2: Physics engine & scenario library ────────────────────────────────
from physiq.engine import PhysIQWorld
from physiq.formats import build_prompt, format_as_json, format_as_ascii, format_as_nl
from physiq.generation import validate_scenario, _task_seed
from physiq.scoring import score_tool_use
from physiq.templates.tool_use import TOOL_USE_TEMPLATES
from physiq.templates import SCENARIO_COUNTS


def _fmt(scenario: dict, fmt: str) -> str:
    """Dispatch to the correct format function."""
    if fmt == 'ascii':
        return format_as_ascii(scenario)
    if fmt == 'nl':
        return format_as_nl(scenario)
    return format_as_json(scenario)


print(f'Tool use templates: {len(TOOL_USE_TEMPLATES)}')

In [ ]:
# ── Cell 3: Generate tool use scenarios ─────────────────────────────────────
MASTER_SEED = 42
task_type = 'tool_use'
counts = SCENARIO_COUNTS[task_type]

task_seed = _task_seed(task_type, MASTER_SEED)
rng = np.random.RandomState(task_seed)

scenarios = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected = []
    attempt = 0
    while len(collected) < target and attempt < target * 3:
        seed = int(rng.randint(0, 2**31))
        template = TOOL_USE_TEMPLATES[attempt % len(TOOL_USE_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            collected.append(s)
    scenarios.extend(collected)
    print(f'  {difficulty}: {len(collected)}/{target}')

print(f'Total tool use scenarios: {len(scenarios)}')

In [ ]:
# ── Cell 4: Build evaluation DataFrame ──────────────────────────────────────
# Multi-turn tasks pass the full scenario JSON so the task function can
# re-simulate world state after each action.
rows = []
for s in scenarios:
    for fmt in ['json', 'ascii', 'nl']:
        s_with_fmt = dict(s)
        s_with_fmt['_format'] = fmt
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'scenario_json':  json.dumps(s_with_fmt),
        })

df_tool_use = pd.DataFrame(rows)
print(df_tool_use.shape)
df_tool_use.head(2)

In [ ]:
# ── Cell 5: Multi-turn conversation helpers ──────────────────────────────────
MAX_TURNS = 10

_PHYSICS_KEYWORDS = {
    'force', 'gravity', 'friction', 'momentum', 'velocity', 'mass',
    'inertia', 'torque', 'lever', 'ramp', 'slope', 'balance', 'weight',
    'support', 'bridge', 'span', 'push', 'angle', 'trajectory', 'arc',
}


def _reasoning_valid(response: str) -> bool:
    """True if the response contains 3+ physics keywords."""
    resp_lower = response.lower()
    hits = sum(1 for kw in _PHYSICS_KEYWORDS if kw in resp_lower)
    return hits >= 3


def _parse_action(response: str):
    """Extract ACTION: line → dict, or None if unparseable."""
    m = re.search(r'^ACTION\s*:\s*(.+)$', response, re.IGNORECASE | re.MULTILINE)
    if not m:
        return None
    action_str = m.group(1).strip()

    # PLACE <type> AT <x> <y> [ANGLE <deg>]
    pm = re.match(
        r'PLACE\s+(\w+)\s+AT\s+([\d.\-]+)\s+([\d.\-]+)(?:\s+ANGLE\s+([\d.\-]+))?',
        action_str, re.IGNORECASE
    )
    if pm:
        return {
            'type': 'place',
            'object_type': pm.group(1),
            'position': [float(pm.group(2)), float(pm.group(3))],
            'angle': float(pm.group(4)) if pm.group(4) else 0.0,
        }

    # PUSH <id> [WITH_]FORCE <n> DIRECTION <deg>
    pu = re.match(
        r'PUSH\s+(\w+)\s+(?:WITH_FORCE|FORCE)\s+([\d.\-]+)\s+DIRECTION\s+([\d.\-]+)',
        action_str, re.IGNORECASE
    )
    if pu:
        return {
            'type': 'push',
            'object_id': pu.group(1),
            'force': float(pu.group(2)),
            'direction': float(pu.group(3)),
        }

    # REMOVE <id>
    rm = re.match(r'REMOVE\s+(\w+)', action_str, re.IGNORECASE)
    if rm:
        return {'type': 'remove', 'object_id': rm.group(1)}

    return None


print('Helpers defined OK')

In [ ]:
# ── Cell 6: Task definition (multi-turn) ────────────────────────────────────
@kbench.task(
    name='PhysIQ Tool Use Planning',
    description=(
        'Use up to 10 actions (PLACE/PUSH/REMOVE) to manipulate a 2D physics '
        'world and achieve the stated goal. '
        'Tests means-end reasoning and creative problem solving.'
    ),
)
def physiq_tool_use(llm: kbench.LLMChat, scenario_json: str) -> float:
    """Multi-turn: propose physics actions to achieve a goal."""
    scenario = json.loads(scenario_json)
    fmt = scenario.pop('_format', 'json')
    world = PhysIQWorld(scenario)
    goal = scenario.get('goal')
    goal_desc = scenario.get('goal_description', str(goal))
    tools = scenario.get('available_tools', [])
    tool_list = '\n'.join(f'  - {t}' for t in tools) if tools else '  (none specified)'

    turns_used = 0
    any_valid_reasoning = False

    prompt = (
        f"You are controlling a 2D physics simulation.\n\n"
        f"GOAL: {goal_desc}\n\n"
        f"AVAILABLE TOOLS:\n{tool_list}\n\n"
        f"CURRENT STATE:\n{_fmt(scenario, fmt)}\n\n"
        f"Available actions:\n"
        f"  PLACE <object_type> AT <x> <y> ANGLE <degrees>\n"
        f"  PUSH <object_id> FORCE <newtons> DIRECTION <degrees>\n"
        f"  REMOVE <object_id>\n\n"
        f"Explain your reasoning, then output your action on a line starting with ACTION:"
    )

    for _turn in range(MAX_TURNS):
        response = llm.prompt(prompt)
        turns_used += 1

        if _reasoning_valid(response):
            any_valid_reasoning = True

        action = _parse_action(response)
        if action is None:
            prompt = (
                "I couldn't parse your action. Use exactly one of:\n"
                "  ACTION: PLACE <type> AT <x> <y> ANGLE <deg>\n"
                "  ACTION: PUSH <id> FORCE <n> DIRECTION <deg>\n"
                "  ACTION: REMOVE <id>\n"
                f"Turns remaining: {MAX_TURNS - turns_used}"
            )
            continue

        result_msg = world.execute_action(action)
        world.simulate(0.5)  # settle physics

        if world.check_goal(goal):
            return score_tool_use(
                goal_achieved=True,
                turns_used=turns_used,
                max_turns=MAX_TURNS,
                reasoning_valid=any_valid_reasoning,
            )

        if turns_used >= MAX_TURNS:
            break

        prompt = (
            f"Action result: {result_msg}\n\n"
            f"UPDATED STATE:\n{_fmt(scenario, fmt)}\n\n"
            f"GOAL: {goal_desc}\n"
            f"Turns remaining: {MAX_TURNS - turns_used}\n\n"
            f"Explain your reasoning, then output your action on a line starting with ACTION:"
        )

    # Max turns: partial credit scaled by progress
    progress = world.measure_progress(goal)
    return score_tool_use(
        goal_achieved=False,
        turns_used=MAX_TURNS,
        max_turns=MAX_TURNS,
        reasoning_valid=any_valid_reasoning,
        progress=progress,
    )

In [ ]:
# ── Cell 7: Dry-run sanity check ─────────────────────────────────────────────
required_cols = {'scenario_json'}
assert required_cols.issubset(df_tool_use.columns)
print('DataFrame columns OK:', list(df_tool_use.columns))
print('Rows:', len(df_tool_use))
print('Difficulties:', df_tool_use.groupby('difficulty').size().to_dict())

s0 = json.loads(df_tool_use.iloc[0]['scenario_json'])
print('Sample scenario ID:', s0.get('id'))
print('Goal:', s0.get('goal'))
print('Available tools:', s0.get('available_tools', [])[:3])

In [ ]:
# ── Cell 8: Evaluation run ───────────────────────────────────────────────────
try:
    llm = kbench.llm
except AttributeError:
    print('No Kaggle LLM available (local run). Skipping evaluation.')
    llm = None

if llm is not None:
    results = []
    for _, row in df_tool_use.iterrows():
        run = physiq_tool_use.run(
            llm=llm,
            scenario_json=row['scenario_json'],
        )
        results.append({
            'scenario_id':    row['scenario_id'],
            'difficulty':     row['difficulty'],
            'representation': row['representation'],
            'score':          run.result,
        })
    df_results = pd.DataFrame(results)
    os.makedirs('../outputs', exist_ok=True)
    df_results.to_csv('../outputs/task4_tool_use_results.csv', index=False)
    print('Mean score:', df_results['score'].mean())
    print(df_results.groupby(['difficulty', 'representation'])['score'].mean().unstack())

In [ ]:
# ── Cell 9: Results analysis ─────────────────────────────────────────────────
if llm is not None and 'df_results' in dir():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_results.groupby('difficulty')['score'].mean().plot(
        kind='bar', ax=axes[0], color=['#4CAF50', '#FF9800', '#F44336'])
    axes[0].set_title('Tool Use Score by Difficulty')
    axes[0].set_ylabel('Mean Score (0-1)')
    axes[0].set_ylim(0, 1)

    df_results.groupby('representation')['score'].mean().plot(
        kind='bar', ax=axes[1], color=['#2196F3', '#9C27B0', '#00BCD4'])
    axes[1].set_title('Tool Use Score by Format')
    axes[1].set_ylabel('Mean Score (0-1)')
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../outputs/task4_tool_use_results.png', dpi=150)
    plt.show()

In [ ]:
# ── Cell 10: Choose this task for the leaderboard ────────────────────────────
%choose physiq_tool_use